In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_annotated/checkpoints/pre_cell_0.pickle

trying: ['dirname']
me:  0
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['Path']
me:  0
trying: ['benchmark_name']
me:  0
trying: ['re']
me:  0
trying: ['filename']
me:  0
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['filenames']
me:  0
Declaring variable dirname
Declaring variable np
Declaring variable pd
Declaring variable Path
Declaring variable benchmark_name
Declaring variable re
Declaring variable filename
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable filenames


In [4]:
%%RecordEvent
%%time
### cell 0 ###

# cell 0 – optimized for cudf
from pathlib import Path

base_input = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input"
mobile_list = []
broadband_list = []

for root, _, files in os.walk(base_input):
    for file in files:
        filepath = Path(root) / file
        # read directly into GPU (cudf), avoid convert_dtypes CPU fallbacks
        df = pd.read_csv(filepath, thousands=',')
        # rename if needed (cudf.rename is GPU)
        df.rename(columns={'Number of Record':'Number of Records'}, inplace=True)
        # extract year & quarter once per file
        yr = int(re.search(r'year_(\d+)_quarter', file).group(1))
        qt = int(re.search(r'quarter_(\d+)\.csv', file).group(1))
        df['Year'] = yr
        df['Quarter'] = qt
        # collect into lists, defer concat
        if 'mobile' in file:
            mobile_list.append(df)
        else:
            broadband_list.append(df)

# single GPU concat per category instead of per‐file
Mobile_df = pd.concat(mobile_list, ignore_index=True)
Broadband_df = pd.concat(broadband_list, ignore_index=True)

print(Broadband_df.shape)
print(Mobile_df.shape)

# GPU astype & sort (assignment, not fallback inplace)
Mobile_df = Mobile_df.astype({'Year':'int64','Quarter':'int64'})
Broadband_df = Broadband_df.astype({'Year':'int64','Quarter':'int64'})
Mobile_df = Mobile_df.sort_values(['Year','Quarter'])
Broadband_df = Broadband_df.sort_values(['Year','Quarter'])

# expand datasets with fewer GPU concats
factor = 10
Mobile_df = pd.concat([Mobile_df] * (factor * 2), ignore_index=True)
Broadband_df = pd.concat([Broadband_df] * factor, ignore_index=True)

Mobile_df.info()
Broadband_df.info()

(2597, 13)
(2487, 13)


<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 49740 entries, 0 to 49739
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               49740 non-null  object
 1   Number of Records  49740 non-null  int64
 2   Devices            49740 non-null  int64
 3   Tests              49740 non-null  int64
 4   Avg. Avg U Kbps    49740 non-null  int64
 5   Avg. Avg D Kbps    49740 non-null  int64
 6   Avg Lat Ms         49740 non-null  int64
 7   Avg. Pop2005       49740 non-null  int64
 8   Rank Upload        49740 non-null  int64
 9   Rank Download      49740 non-null  int64
 10  Rank Latency       49740 non-null  int64
 11  Year               49740 non-null  int64
 12  Quarter            49740 non-null  int64
dtypes: int64(12), object(1)
memory usage: 5.2+ MB
<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 25970 entries, 0 to 25969
Data columns (total 13 columns):
 #   Column             Non-Null Cou

In [5]:
%Checkpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_0_try_0.pickle

migration speed (bps): 834759425.4949492
---------------------------
variables to migrate:
file 80
factor 28
files 248
filepath 88
yr 28
re 72
mobile_list 273970
filenames 248
root 141
BENCHMARKS_TO_PATHS 2272
filename 80
benchmark_name 63
Path 904
Mobile_df 5474844
dirname 141
Broadband_df 2866634
df 24651
qt 28
pd 72
broadband_list 286891
base_input 88
np 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/rewritten/o4_mini_high_small/checkpoints/post_cell_0_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['BENCHMARKS_TO_PATHS', 'benchmark_name', 'Path']
Active variables []
Intermediate variables ['yr', 'Mobile_df', 'df', 'file', 'broadband_list', 'qt', 'files', 'mobile_list', 'filepath', 'base_input', 'Broadband_df', 'factor', 'root']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/opt_cell_exec_info_0_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[0], f)


In [8]:
opt_output = Out.get(4)

In [9]:
Mobile_df_opt = Mobile_df
Broadband_df_opt = Broadband_df
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_0.pickle
assert compare_df(Mobile_df_opt, Mobile_df)
assert compare_df(Broadband_df_opt, Broadband_df)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['filename', 'meta_info']
me:  1
me:  1
trying: ['filename', 'meta_info']
me:  1
me:  1
trying: ['filenames']
me:  1
trying: ['Path']
me:  0
trying: ['data']
me:  1
trying: ['re']
me:  0
trying: ['dirname']
me:  0
trying: ['Mobile_df']
me:  1
trying: ['factor']
me:  1
trying: ['pd']
me:  0
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['Broadband_df']
me:  1
trying: ['orig_output']
me:  2
trying: ['benchmark_name']
me:  0
trying: ['np']
me:  0
trying: ['col_name_corrections']
me:  1
Declaring variable Path
Declaring variable re
Declaring variable dirname
Declaring variable pd
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable benchmark_name
Declaring variable np
Declaring variable filename
Declaring variable meta_info
Declaring variable filename
Declaring variable meta_info
Declaring variable filenames
Declaring variable data
Declaring variable Mobile_df
Declaring variable factor
Declaring variable Broadband_df
Declaring variable col_name_corrections
Declaring variable 

ValueError: Data type mismatch in column 'Name': Rewritten code is object and original code is string